## 子图

#### [持久化](https://docs.langchain.com/oss/python/langgraph/use-subgraphs#subgraph-persistence)

子图 `.compile(checkpointer=...)` 决定子图内部状态在多次父图调用之间是否保留：

- **`None`（默认，按调用）**：每次调用从空状态开始；继承父图的 checkpointer，支持单次调用内的中断与失败恢复。
- **`True`（按线程）**：同一 `thread_id` 下状态累积，每次调用接着上次继续。
- **`False`（无状态）**：完全不做检查点，等同普通函数调用。

父图必须带 checkpointer，以上能力才生效。

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt


# ===== 子图：第一次询问名字，之后沿用记住的名字 =====
class SubgraphState(TypedDict):
    name: str
    greeting: str

def ask_name(state: SubgraphState):
    name = state.get("name")
    if not name:
        name = interrupt("请问你的名字？")
    return {"name": name, "greeting": f"你好，{name}！"}


sub_builder = StateGraph(SubgraphState)
sub_builder.add_node("ask_name", ask_name)
sub_builder.add_edge(START, "ask_name")
sub_builder.add_edge("ask_name", END)
subgraph = sub_builder.compile(checkpointer=True)  # 按线程记住子图内部状态


# ===== 父图 =====
class ParentState(TypedDict):
    greeting: str

builder = StateGraph(ParentState)
builder.add_node("subgraph_node", subgraph)
builder.add_edge(START, "subgraph_node")
builder.add_edge("subgraph_node", END)
graph = builder.compile(checkpointer=InMemorySaver())  # 父图必须带 checkpointer

config = {"configurable": {"thread_id": "8-3-1"}}

# 第一次调用：子图中断询问名字
result = graph.invoke({"greeting": ""}, config=config)
print(result)

# 恢复中断，把名字交给子图记录
result = graph.invoke(Command(resume="Max"), config=config)
print(result)

# 再次调用：不再询问，直接叫出记住的名字
result = graph.invoke({"greeting": ""}, config=config)
print(result)